In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import numpy as np
from scipy.sparse import csr_matrix
import csv
import random
import pickle
import json
import pandas as pd



In [9]:
import pandas as pd
def get_first_value(s):
    lst = eval(s)
    return lst[0]
event_file = r"./processed_events_shuffled.dat"
data = pd.read_csv(event_file, sep='\t', header=None, names=['user', 'timestamp', 'armpool'])
data= data.iloc[1:,:]

data["picked_item"] = data['armpool'].apply(get_first_value)
data 



,user,timestamp,armpool,picked_item
1,46,1283292000000,"[295, 7288, 11206, 13034, 9964, 5604, 11315, 1...",295
2,1326,1228086000000,"[288, 4104, 12805, 14978, 14340, 13225, 16644,...",288
3,1535,1272664800000,"[7123, 8876, 10576, 4442, 7556, 12113, 10208, ...",7123
4,650,1191189600000,"[3902, 803, 11498, 5605, 15562, 3650, 1583, 17...",3902
5,1769,1246399200000,"[1243, 2832, 870, 14087, 699, 18391, 14142, 10...",1243
...,...,...,...,...
96729,411,1214863200000,"[475, 8363, 13785, 17907, 14570, 6233, 15262, ...",475
96730,357,1243807200000,"[6388, 18574, 6771, 11131, 4622, 1337, 5256, 1...",6388
96731,1822,1277935200000,"[9155, 9200, 17404, 17234, 1309, 3347, 8926, 6...",9155
96732,1217,1291158000000,"[707, 18609, 6835, 8788, 2637, 13456, 12002, 9...",707


In [13]:
data2 = data[["user","picked_item"]]
data2= data2.reset_index(drop=True)
data2

,user,picked_item
0,46,295
1,1326,288
2,1535,7123
3,650,3902
4,1769,1243
...,...,...
96728,411,475
96729,357,6388
96730,1822,9155
96731,1217,707


In [20]:
import numpy as np
FeatureVectors = {}
with open('./Arm_FeatureVectors_2.dat', "r") as f:
    for line in f:
        line = line.split("\t")
        vec = line[1].strip("[]").strip("\n").split(";") #[0] 是item id [1]是arm feature
        FeatureVectors[int(line[0])] = np.array(vec).astype(np.float)
# FeatureVectors



d:\SoftwareFamily\Anaconda\envs\nlplab\lib\site-packages\ipykernel_launcher.py:7: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  import sys


{52: array([ 0.10568702, -0.06818229, -0.41204546,  0.35458666, -0.04224088,
         0.01919637,  0.10927453,  0.0839815 ,  0.11687149, -0.01779011,
         0.05488165,  0.03145848, -0.10723706, -0.02694798, -0.10281454,
         0.09930332,  0.0893081 , -0.15731957, -0.14839965,  0.11303263,
         0.00412588,  0.10442615, -0.17321055, -0.12906204, -0.01506565]),
 63: array([ 0.0875718 , -0.03493151, -0.16046736,  0.03671298,  0.09810837,
         0.14123745,  0.37590639, -0.01531431,  0.0578995 , -0.13208602,
        -0.19649849,  0.13972472, -0.15252865, -0.04887264,  0.03046871,
        -0.04110974,  0.05374018, -0.15685045,  0.02506383, -0.06176694,
        -0.00740979,  0.08900594, -0.04204989,  0.03754526,  0.08987929]),
 73: array([ 0.1302909 , -0.03727091, -0.16032599,  0.10376093,  0.00551067,
         0.03821518,  0.2346223 ,  0.10629966,  0.13732067, -0.04119535,
        -0.01649755,  0.17309532, -0.16922014,  0.0339166 , -0.06441467,
         0.20138471,  0.00197677, -

In [21]:
data2["vector"] = data2["picked_item"].apply(lambda x: FeatureVectors[int(x)])

data2



,user,picked_item,vector
0,46,295,"[0.198105271095, -0.120441564893, -0.513285157..."
1,1326,288,"[0.264864077083, -0.151376289625, -0.591367725..."
2,1535,7123,"[0.026714469914, -0.0102648091517, -0.05784553..."
3,650,3902,"[0.335463886333, -0.132924508804, -0.230038553..."
4,1769,1243,"[0.303273924208, -0.152522102457, -0.367109724..."
...,...,...,...
96728,411,475,"[0.0740732304714, -0.0518889850447, -0.4298387..."
96729,357,6388,"[0.0682554059354, -0.0538198237039, -0.4372134..."
96730,1822,9155,"[0.0124508073157, -0.00227429826354, -0.003006..."
96731,1217,707,"[0.29812985971, 0.781472297137, -0.09547337719..."


In [36]:
mean_vectors  = data2.groupby("user")["vector"].mean()
result_df = pd.DataFrame({'user': mean_vectors.index, 'mean_vector': mean_vectors.values})
result_df

# user_vectors

,user,mean_vector
0,1,"[0.14651610408492, -0.03524186513439, -0.13151..."
1,10,"[0.2323631672838095, -0.08155480954594285, -0...."
2,100,"[0.38110215476591164, 0.028847463113691175, 0...."
3,1000,"[0.5010557456564286, 0.012765960823234642, 0.0..."
4,1001,"[0.20597415220017776, -0.09423752447720299, -0..."
...,...,...
1887,995,"[0.33490970290478894, -0.06003423397522645, -0..."
1888,996,"[0.394614651284, -0.10621367388695, 0.04733795..."
1889,997,"[0.38380028034067065, -0.019712678101738424, 0..."
1890,998,"[0.13048619508163245, -0.019063641214482705, -..."


In [37]:
result_df[['u{}'.format(i) for i in range(len(result_df['mean_vector'][0]))]] = result_df['mean_vector'].apply(pd.Series)
result_df = result_df.drop('mean_vector', axis=1)
result_df


,user,u0,u1,u2,u3,u4,u5,u6,u7,u8,...,u15,u16,u17,u18,u19,u20,u21,u22,u23,u24
0,1,0.146516,-0.035242,-0.131514,0.083956,0.086915,0.030891,0.187623,-0.029845,0.020943,...,0.055610,0.026923,-0.078993,0.021650,-0.031373,0.022851,0.047545,-0.056427,0.031172,0.005416
1,10,0.232363,-0.081555,-0.248638,0.027093,0.095547,-0.101298,-0.084566,-0.090268,-0.071834,...,-0.000021,-0.011883,0.037535,0.044342,-0.017238,-0.004193,-0.038553,-0.014560,-0.027404,-0.009196
2,100,0.381102,0.028847,0.005632,0.007148,0.054034,-0.037941,-0.020789,-0.013469,-0.012093,...,-0.000951,0.011067,-0.015101,-0.007707,0.001102,0.059045,0.009473,0.027300,-0.004653,0.004550
3,1000,0.501056,0.012766,0.024526,-0.012577,-0.021955,-0.083902,-0.030951,0.041419,0.084193,...,0.010787,0.000394,-0.002209,0.022088,-0.003369,-0.003611,0.010476,-0.016955,0.001728,-0.011246
4,1001,0.205974,-0.094238,-0.280693,-0.112118,0.095212,-0.180953,-0.035970,-0.009452,-0.083246,...,-0.018163,-0.012631,0.000735,-0.075945,0.053570,-0.103092,0.118748,0.049006,0.156359,0.039533
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1887,995,0.334910,-0.060034,-0.046841,-0.021949,0.031356,-0.115047,-0.005808,0.020633,0.003952,...,0.001726,-0.002547,-0.026334,0.040394,0.002703,-0.024378,-0.015015,-0.016494,-0.023620,-0.004378
1888,996,0.394615,-0.106214,0.047338,-0.091694,-0.255788,0.204071,-0.065478,-0.001207,-0.017908,...,0.037049,-0.030204,-0.037304,0.024258,0.002283,0.009137,0.001110,-0.003936,-0.019321,0.023620
1889,997,0.383800,-0.019713,0.005815,0.021731,0.291234,0.071020,-0.125398,0.076468,-0.000342,...,0.010982,-0.008682,-0.015261,0.003954,0.012496,0.041983,0.006839,0.014767,-0.024760,0.030195
1890,998,0.130486,-0.019064,-0.101496,0.003898,0.115576,0.008739,0.104791,-0.011906,0.080871,...,0.100098,-0.005045,-0.015133,0.026074,0.001952,0.003328,0.057320,-0.014684,0.006163,-0.002116


In [ ]:
result_df.to_csv('./lastfm_User_Glovemean_FeatureVectors.csv', sep=' ', index=False, header=True)

In [ ]:
#数据读入预处理
import pandas as pd
event_file = r"./Arm_FeatureVectors_2.dat"
data = pd.read_csv('data.txt', sep='\t', header=None, names=['user', 'timestamp', 'armpool'])

# user_id_list, user_documents_list, user_documents_dict = data_process_user(event_file) # 返回前一周的user列表，元素是venue id

# # 初始化 TfidfVectorizer
# vectorizer = TfidfVectorizer()
# # 计算 TF-IDF
# tfidf_matrix = vectorizer.fit_transform(user_documents_list) # 每一行代表一个文章，每一列代表一个单词
# #将稀疏矩阵转化为密集矩阵再转化为array
# dense_matrix = np.asarray(csr_matrix.todense(tfidf_matrix))
# # 创建 PCA 对象，指定降维后的维度为 25
# pca = PCA(n_components=25)
# # 对数据矩阵进行拟合和转换
# X_projected = pca.fit_transform(dense_matrix)
# #将降维后的向量与user id 匹配
# filename = 'User_FeatureVectors.dat'
# with open(filename, 'w', newline='') as f:
#     for i in range(X_projected.shape[0]):
#         new_line = str(user_id_list[i]) + '\t' + ';'.join(str(X_projected[i, :]).strip('[').strip(']').split()) + '\n'
#         f.write(new_line)

# 读取没有列名的CSV文件
data = pd.read_csv("./POI_Glove_FeatureVectors.csv", header=0, sep=' ')
FeatureVectors = {}
for index, row in data.iterrows():
    key = int(row[0])
    value = row[1:].tolist()
    FeatureVectors[key] = np.array(value)
# cat_id_map_dict = {}

# with open("./dataset/foursquare/cat_id_map_dict.json", 'r') as f:
#     cat_id_map_dict = json.load(f)
with open("./cat_id_map_dict.json", 'r') as f:
    cat_id_map_dict = json.load(f)

users = []
userfeatures = []
for user in user_documents_dict:
    list = user_documents_dict[user]
    newlist =  [cat_id_map_dict.get(item) for item in list]
    featurelist = [FeatureVectors.get(item) for item in newlist]
    featuremean = np.mean(featurelist,axis = 0)
    users.append(user)
    userfeatures.append(featuremean)
users_glove_feature = pd.DataFrame({'uid': users, 'featuremean': userfeatures})
# 展开userfeatures列
users_glove_feature_expanded = users_glove_feature["featuremean"].apply(pd.Series)
users_glove_feature_expanded.columns = [f"u{i}" for i in range(len(users_glove_feature_expanded.columns))]
# 合并展开后的列和原始列
result_df = pd.concat([users_glove_feature.drop("featuremean", axis=1), users_glove_feature_expanded], axis=1)
result_df

